# Cubism on a cube tessellation: a quick demo

The next piece: Tristan Miller's Cube Tessellation, folded from an 8 × 8 grid, with one subject painted from three viewpoints.

**The geometry.** Each cube shows three faces. Tilt the model until you look straight along one cube axis, about 55° off the paper's normal, and two of the three faces turn edge-on and vanish, while the third shows as true squares. Those squares tile one continuous picture. So the model holds three pictures, one per axis, and every visible bit of paper belongs to exactly one of them.

**The piece.** One guitar, from three viewpoints: front, side and above, one per axis. Tilted, each viewpoint resolves on its own. Head-on, all three are cut into planes and interleaved: the guitar seen from three sides at once, which is what cubism painted.

**This demo is the cheapest version.** It's virtual: no paper, no visibility map, and the finished model's 25 faces as ideal squares (step 18 of the diagrams), leaving out the flat triangles at its rim. Since no paper is shared between the three pictures, each is generated on its own with plain DeepFloyd, no multi-view sampling needed, then painted onto the faces and rendered. That's enough to see whether it reads before we build the real thing.

## Setup

The geometry and renderer. No GPU needed.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image, ImageDraw, ImageFont

# The finished model's faces (step 18 of the diagrams). Start from cubes at lattice points of the
# plane x + y + z = 0: head-on, along (1, 1, 1), they tile as tumbling blocks, each showing its +x, +y
# and +z faces. Nine cubes give rows of 1, 2, 3, 2, 1. The model differs at the bottom and the sides:
# - The bottom is a hollow corner, not a cube: drop the bottom cube's two side faces, and the outer
#   side faces of the two cubes above it.
# - Each side has a wall rising from the outer cube's top to the second row.
CUBES = [(a, b, -(a + b)) for a in range(3) for b in range(3)]
DROP = {(0, (3, 2, -4)), (1, (2, 3, -4)), (0, (3, 1, -3)), (1, (1, 3, -3))}   # (axis, corner)
WALLS = [(1, (2, 0, -1)), (0, (0, 2, -1))]                  # (axis, corner)
AXES = ("x", "y", "z")
E = np.eye(3)
HEAD_ON = np.ones(3) / np.sqrt(3)
HEAD_ON_UP = np.array([-1.0, -1.0, 2.0]) / np.sqrt(6)      # screen up when seen head-on
LIT = (0.8, 0.9, 1.0)                                      # x, y, z faces lit from above and to the right


def face(k, corner):
    i, j = [m for m in range(3) if m != k]
    return (k, np.array(corner, float), E[i], E[j])


def faces():
    """Every visible face: (axis it faces, corner, edge 1, edge 2)."""
    out = [face(k, np.add(p, E[k])) for p in CUBES for k in range(3)]
    out = [f for f in out if (f[0], tuple(f[1].astype(int))) not in DROP]
    return out + [face(k, c) for k, c in WALLS]


def camera(t=0.0, axis=2):
    """Toward-viewer direction and screen right and up, tilted a fraction t of the way from head-on to
    looking straight along an axis. Up stays as close to head-on's up as it can, as when tilting a
    model in your hands."""
    a, b = HEAD_ON, E[axis]
    ang = np.arccos(a @ b)
    d = (np.sin((1 - t) * ang) * a + np.sin(t * ang) * b) / np.sin(ang)
    up = HEAD_ON_UP - (HEAD_ON_UP @ d) * d
    up /= np.linalg.norm(up)
    return d, np.cross(up, d), up


def bounds(cam, fs):
    _, r, u = cam
    pts = np.array([c + s * e1 + q * e2 for _, c, e1, e2 in fs for s in (0, 1) for q in (0, 1)])
    return (pts @ r).min(), (pts @ r).max(), (pts @ u).min(), (pts @ u).max()


def render(pictures=None, t=0.0, axis=2, size=512, margin=0.06, bg=1.0, light=None, textures=None):
    """The model seen at a tilt t toward an axis. pictures: one (H, W, 3) array in [0, 1] per axis,
    each covering (center-cropped to) the box around that axis's faces as seen straight along it.
    light: a brightness per axis, to see the faces as lit paper, e.g. LIT.
    textures: instead of pictures, one (n, n, 3) array per face, rows along its edge 2."""
    fs = faces()
    d, r, u = cam = camera(t, axis)
    x0, x1, y0, y1 = bounds(cam, fs)
    span = max(x1 - x0, y1 - y0) * (1 + 2 * margin)
    X, Y = np.meshgrid((x0 + x1) / 2 + ((np.arange(size) + 0.5) / size - 0.5) * span,
                       (y0 + y1) / 2 - ((np.arange(size) + 0.5) / size - 0.5) * span)
    out = np.full((size, size, 3), bg)
    depth = np.full((size, size), -np.inf)
    cams = [camera(1.0, k) for k in range(3)]
    boxes = [bounds(cams[k], [f for f in fs if f[0] == k]) for k in range(3)]
    for i, (k, c, e1, e2) in enumerate(fs):
        if E[k] @ d < 1e-6:                                # seen edge-on
            continue
        inv = np.linalg.inv(np.array([[e1 @ r, e2 @ r], [e1 @ u, e2 @ u]]))
        qx, qy = X - c @ r, Y - c @ u
        s, q = inv[0, 0] * qx + inv[0, 1] * qy, inv[1, 0] * qx + inv[1, 1] * qy
        P = c + s[..., None] * e1 + q[..., None] * e2
        z = P @ d
        hit = (s >= 0) & (s <= 1) & (q >= 0) & (q <= 1) & (z > depth)
        if textures is not None:
            n = textures[i].shape[0]
            si, qi = (s * n).astype(int).clip(0, n - 1), (q * n).astype(int).clip(0, n - 1)
            out[hit] = textures[i][qi[hit], si[hit]] * (1 if light is None else light[k])
            depth[hit] = z[hit]
            continue
        # Each face shows the part of its axis's picture that it covers when seen along that axis.
        _, rk, uk = cams[k]
        bx0, bx1, by0, by1 = boxes[k]
        pic = pictures[k]
        H, W = pic.shape[:2]
        scale = max(W / (bx1 - bx0), H / (by1 - by0))        # cover the box, cropping the overflow
        px = ((P @ rk - (bx0 + bx1) / 2) * scale + W / 2).astype(int).clip(0, W - 1)
        py = (((by0 + by1) / 2 - P @ uk) * scale + H / 2).astype(int).clip(0, H - 1)
        out[hit] = pic[py[hit], px[hit]] * (1 if light is None else light[k])
        depth[hit] = z[hit]
    return out


def placeholder(text, color, size=256):
    """A stand-in picture: a letter on stripes, to check the geometry without a GPU."""
    img = Image.new("RGB", (size, size), color)
    draw = ImageDraw.Draw(img)
    for i in range(-size, size, 24):
        draw.line([(i, 0), (i + size, size)], fill=tuple(int(v * 0.8) for v in color), width=6)
    try:
        font = ImageFont.truetype("DejaVuSans-Bold.ttf", size // 3)
    except OSError:
        font = ImageFont.load_default(size // 3)
    draw.text((size / 2, size / 2), text, fill="black", font=font, anchor="mm")
    return np.asarray(img, float) / 255


def plot_tilts(pictures, titles, tilts=(0, 0.25, 0.5, 0.75, 1), size=256, light=None, textures=None):
    """One row per axis: the model tilted from head-on to looking straight along that axis."""
    fig, axes = plt.subplots(3, len(tilts), figsize=(3 * len(tilts), 9.3))
    for k in range(3):
        for j, t in enumerate(tilts):
            axes[k, j].imshow(render(pictures, t, k, size, light=light, textures=textures))
            axes[k, j].set_title("head-on" if t == 0 else f"{titles[k]}" if t == 1 else f"{t:.0%} toward {AXES[k]}",
                                 fontsize=10)
            axes[k, j].axis("off")
    fig.tight_layout()
    return fig


def save_gif(pictures, path, frames=14, size=360, hold=6, light=None, textures=None):
    """Head-on, tilt to x and back, then y, then z."""
    ease = lambda s: (1 - np.cos(np.pi * s)) / 2
    imgs = []
    for k in range(3):
        ts = [ease(i / frames) for i in range(frames + 1)]
        seq = [0.0] * 2 + ts + [1.0] * hold + ts[::-1]
        imgs += [Image.fromarray((render(pictures, t, k, size, light=light, textures=textures) * 255).astype(np.uint8)) for t in seq]
    imgs[0].save(path, save_all=True, append_images=imgs[1:], duration=60, loop=0)

## The three windows

Each axis's picture is seen through a window: the outline of that axis's faces. Stand-in letters show the shapes and how the tilt goes. Head-on (left), each cube shows a piece of all three pictures. Tilted all the way (right), one picture shows whole, and the other faces have vanished edge-on.

The x and y windows are tall zigzags and the z window is a diamond, so put the tall viewpoints on x and y.

In [ ]:
STAND_INS = [placeholder("X", (230, 120, 90)), placeholder("Y", (90, 160, 230)), placeholder("Z", (120, 200, 110))]
plot_tilts(STAND_INS, ("x", "y", "z"))
plt.show()

## A 3D guitar, painted flat (no GPU)

Prompts can't reliably ask for a viewpoint, so here the viewpoints come from a 3D guitar instead, built from simple shapes. The piece works in two layers:

- **Head-on is the key: the guitar's outline.** Each face shows guitar only where a line from it straight toward the head-on viewer meets a guitar lying flat in the paper, with a dark contour at the edge. Head-on, those pieces line up into one clear guitar outline, whatever is inside them.
- **Inside, every face is its own fragment:** a close-up of one part of the guitar (soundhole, bridge, a bout, the waist, the neck or the headstock), seen roughly from the face's axis, each from its own angle, framing and roll. The fragments come from a second guitar turned so that **x sees its front, y its side, and z looks down along its neck.** Tilted toward any axis, you see a jumble of fragments: front views on x, side views on y, views down the neck on z.
- **Outside the outline, a cubist still life:** a wall, Braque's wood-grain panel, a table edge, a newspaper with stencilled letters, sheet music, a bottle and a glass. It's drawn flat in head-on screen space like the outline, so head-on it reads as one setting and tilted it scatters too. Its tones are pale, so the guitar stays the strongest shape, and each face's tone shifts slightly so it reads as a plane.

The painting is deliberately plain: flat tones in Braque's ochres and grays, three tones of shading per part so each plane reads as one facet, and dark outlines. Dark gray fills where a fragment misses the guitar. It tests the composition before any diffusion. `LIT` shades the cube faces as lit paper would be.

In [ ]:
# A guitar in its own frame, in cube edges: u along the neck toward the head, v across the body, w out
# of the top. The body is two overlapping ellipses (the bouts), 0.5 thick.
EMPTY, BODY, NECK, HEAD, BRIDGE = range(5)


def guitar(u, v, w):
    """The part at each point, or EMPTY."""
    m = np.zeros(np.shape(u), np.int8)
    lower = ((u + 1.2) / 1.0) ** 2 + (v / 1.0) ** 2 <= 1
    upper = ((u - 0.15) / 0.75) ** 2 + (v / 0.78) ** 2 <= 1
    m[(lower | upper) & (np.abs(w) <= 0.25)] = BODY
    m[(u > 0.7) & (u < 3.0) & (np.abs(v) < 0.14) & (w > -0.05) & (w < 0.25)] = NECK
    m[(u >= 3.0) & (u < 3.7) & (np.abs(v) < 0.2 + 0.1 * (u - 3)) & (w > 0.0) & (w < 0.15)] = HEAD
    m[(np.abs(u + 1.25) < 0.08) & (np.abs(v) < 0.42) & (w > 0.2) & (w < 0.32)] = BRIDGE
    return m


# Braque's palette: ochres, browns and grays.
COLORS = {"top": (0.82, 0.64, 0.38), "side": (0.56, 0.37, 0.21), "back": (0.62, 0.45, 0.28),
          "neck": (0.47, 0.31, 0.18), "fretboard": (0.2, 0.15, 0.11), "head": (0.32, 0.22, 0.13),
          "bridge": (0.2, 0.15, 0.11), "hole": (0.07, 0.06, 0.05), "rosette": (0.38, 0.3, 0.2),
          "string": (0.92, 0.9, 0.82), "background": (0.74, 0.72, 0.66), "shadow": (0.42, 0.37, 0.31)}
LIGHT = np.array([-0.3, 0.2, 1.0]) / np.linalg.norm([-0.3, 0.2, 1.0])   # from the upper left, head-on


def paint(part, u, v, w, n_world, frame):
    """The color of each hit: the part's color, the top's painted details, and flat shading in a few
    tones, so each plane reads as one facet."""
    U, V, W = frame
    out = np.zeros(part.shape + (3,))
    name = np.full(part.shape, "background", object)
    name[part == BODY] = "side"
    name[(part == BODY) & (w > 0.12)] = "top"
    name[(part == BODY) & (w < -0.12)] = "back"
    name[part == NECK] = "neck"
    name[(part == NECK) & (w > 0.12)] = "fretboard"
    name[part == HEAD] = "head"
    name[part == BRIDGE] = "bridge"
    top = name == "top"
    r = np.hypot(u + 0.3, v)
    name[top & (r < 0.36)] = "rosette"
    name[top & (r < 0.3)] = "hole"
    # Six strings from the bridge to the head, spreading toward the bridge.
    on_top = np.isin(name, ["top", "hole", "rosette", "fretboard", "bridge"]) & (w > 0.12)
    spread = np.interp(u, [-1.25, 3.0], [0.1, 0.06])
    for i in np.linspace(-1, 1, 6):
        name[on_top & (u > -1.25) & (u < 3.0) & (np.abs(v - i * spread) < 0.012)] = "string"
    for key, color in COLORS.items():
        out[name == key] = color
    shade = np.clip(n_world @ LIGHT, 0, 1)
    shade = np.round(shade * 3) / 3                   # three tones: flat facets
    hit = part != EMPTY
    out[hit] *= (0.55 + 0.45 * shade[hit])[:, None]
    return out, name


def cast(P, d, frame, center, scale, depth=8.0, steps=300):
    """Look at the guitar along direction d (toward the viewer) through a grid of points P (H, W, 3):
    each pixel shows the first part of the guitar on the line through its point. Returns a picture."""
    H, Wd = P.shape[:2]
    U, V, W = frame
    s = np.linspace(depth, -depth, steps)             # from the viewer's side inward
    first = np.zeros((H, Wd), int)
    hit = np.zeros((H, Wd), bool)
    base = P - center
    for r in range(0, H, 16):                         # a band of rows at a time, to save memory
        Q = (base[r:r + 16, :, None] + s[:, None] * d) / scale
        occupied = guitar(Q @ U, Q @ V, Q @ W) != EMPTY
        first[r:r + 16] = np.argmax(occupied, axis=2)
        hit[r:r + 16] = occupied.any(axis=2)
    # Refine each hit between the last empty step and the first full one, so flat parts come out flat.
    lo, hi = s[np.maximum(first - 1, 0)], s[first]
    for _ in range(10):
        mid = (lo + hi) / 2
        Q = (base + mid[..., None] * d) / scale
        inside = guitar(Q @ U, Q @ V, Q @ W) != EMPTY
        hi, lo = np.where(inside, mid, hi), np.where(inside, lo, mid)
    h = np.where(hit, hi, -depth)                     # height toward the viewer
    S = base + h[..., None] * d                       # the surface hit, relative to the center
    Q = S / scale
    u, v, w = Q @ U, Q @ V, Q @ W
    part = np.where(hit, guitar(u, v, w), EMPTY)
    # Normals from the surface's slope across the grid, turned toward the viewer.
    n = np.cross(np.gradient(S, axis=1), np.gradient(S, axis=0))
    n /= np.linalg.norm(n, axis=-1, keepdims=True) + 1e-12
    n *= np.sign(n @ d)[..., None]
    pic, names = paint(part, u, v, w, n, frame)
    # Outlines where the part or the height changes, like Braque's contours.
    code = np.unique(names, return_inverse=True)[1].reshape(names.shape)
    edge = np.zeros(code.shape, bool)
    for a in (0, 1):
        e = (np.diff(code, axis=a) != 0) | (np.abs(np.diff(h, axis=a)) > 0.15)
        if a == 0:
            edge[:-1] |= e; edge[1:] |= e
        else:
            edge[:, :-1] |= e; edge[:, 1:] |= e
    pic[edge] *= 0.35
    return pic


def frame_from(normal, along):
    """An orthonormal guitar frame: w along normal, u as close to along as it can."""
    W = np.asarray(normal, float); W /= np.linalg.norm(W)
    U = np.asarray(along, float) - (np.asarray(along, float) @ W) * W; U /= np.linalg.norm(U)
    return U, np.cross(W, U), W


def directions(jumble, roll, seed=0):
    """A viewing direction and a roll per face. A face of axis k looks along its axis tilted by an angle
    between jumble[k] / 2 and jumble[k] (degrees), toward a random side, and is rolled by up to roll."""
    rng = np.random.default_rng(seed)
    out = []
    for k, c, e1, e2 in faces():
        tilt = np.radians(rng.uniform(0.5, 1.0) * jumble[k])
        turn = rng.uniform(0, 2 * np.pi)
        d = np.cos(tilt) * E[k] + np.sin(tilt) * (np.cos(turn) * e1 + np.sin(turn) * e2)
        out.append((d, np.radians(rng.uniform(-roll, roll))))
    return out


def covers(P, d, frame, center, scale, depth=8.0, steps=200):
    """Whether the line through each point of P along d meets the guitar."""
    U, V, W = frame
    s = np.linspace(-depth, depth, steps)
    Q = (P[..., None, :] + s[:, None] * d - center) / scale
    return (guitar(Q @ U, Q @ V, Q @ W) != EMPTY).any(axis=-1)


# Parts of the guitar a fragment can center on, in the guitar's frame (u, v, w).
FEATURES = {"soundhole": (-0.3, 0, 0.25), "bridge": (-1.25, 0, 0.3), "lower bout": (-1.5, 0.55, 0.2),
            "upper bout": (0.2, -0.45, 0.2), "waist": (-0.55, 0.75, 0.0), "neck": (1.8, 0, 0.25),
            "headstock": (3.3, 0, 0.1)}


def still_life(size=1024):
    """A cubist still life to sit behind the guitar, drawn flat in head-on screen space: a wall, a
    wood-grain panel, a table, a newspaper with stencilled letters, sheet music, a bottle and a
    glass, in pale tones so the guitar's outline stays the strongest shape. Returns the picture and
    the head-on box it spans (x0, y0, side)."""
    cam = camera(0.0)
    x0, x1, y0, y1 = bounds(cam, faces())
    side = max(x1 - x0, y1 - y0)
    cx, cy = (x0 + x1) / 2, (y0 + y1) / 2
    img = Image.new("RGB", (size, size), (199, 196, 184))                       # the wall
    draw = ImageDraw.Draw(img)
    px = lambda fx, fy: (fx * size, fy * size)                                 # fractions of the box
    poly = lambda pts, **kw: draw.polygon([px(*p) for p in pts], **kw)
    # The model covers x 0.13 to 0.87 and y 0.21 to 0.79 of the box, plus a tower above and below the
    # middle (x 0.38 to 0.62), and the guitar runs down the middle.
    # Wood-grain panel, upper right: Braque's faux bois.
    poly([(0.6, 0.2), (0.9, 0.24), (0.88, 0.5), (0.6, 0.47)], fill=(196, 160, 112))
    for i in range(10):
        pts = [px(0.6 + 0.28 * t, 0.22 + i * 0.026 + 0.01 * np.sin(9 * t + i) + 0.035 * t) for t in np.linspace(0, 1, 30)]
        draw.line(pts, fill=(160, 122, 80), width=3)
    # Newspaper, upper left, with stencilled letters.
    poly([(0.16, 0.31), (0.39, 0.27), (0.41, 0.47), (0.14, 0.5)], fill=(222, 218, 200))
    for i in range(5):
        y = 0.4 + i * 0.016
        draw.line([px(0.17, y + 0.01), px(0.38, y - 0.025)], fill=(150, 146, 136), width=2)
    try:
        font = ImageFont.truetype("DejaVuSans-Bold.ttf", size // 17)
    except OSError:
        font = ImageFont.load_default(size // 17)
    draw.text(px(0.2, 0.32), "JOU", fill=(40, 36, 32), font=font)
    # The table: a tilted top across the lower half, with a dark front edge.
    poly([(0.0, 0.57), (1.0, 0.51), (1.0, 0.73), (0.0, 0.79)], fill=(176, 150, 118))
    poly([(0.0, 0.79), (1.0, 0.73), (1.0, 1.0), (0.0, 1.0)], fill=(120, 100, 80))
    draw.line([px(0.0, 0.79), px(1.0, 0.73)], fill=(50, 42, 34), width=6)
    # Sheet music on the table, lower left.
    poly([(0.12, 0.6), (0.4, 0.57), (0.42, 0.75), (0.14, 0.78)], fill=(236, 232, 214))
    for staff in range(3):
        for line in range(5):
            y = 0.62 + staff * 0.05 + line * 0.007
            draw.line([px(0.14, y + 0.004), px(0.4, y - 0.024)], fill=(70, 66, 60), width=2)
    for i in range(8):
        fx = 0.17 + i * 0.028
        fy = 0.62 + (i % 3) * 0.05 + 0.012 - (fx - 0.14) * 0.11 + 0.005 * ((i * 7) % 4)
        draw.ellipse([px(fx - 0.006, fy - 0.0045), px(fx + 0.006, fy + 0.0045)], fill=(40, 36, 32))
    # A bottle on the table, right, and a glass drawn in outline, left.
    poly([(0.68, 0.68), (0.68, 0.5), (0.71, 0.45), (0.71, 0.37), (0.75, 0.37), (0.75, 0.45), (0.78, 0.5), (0.78, 0.67)],
         fill=(96, 108, 92), outline=(40, 44, 38))
    draw.line([px(0.26, 0.5), px(0.28, 0.61), px(0.34, 0.6), px(0.36, 0.49)], fill=(60, 56, 50), width=4)
    draw.ellipse([px(0.26, 0.475), px(0.36, 0.51)], outline=(60, 56, 50), width=4)
    return np.asarray(img, float) / 255, (cx - side / 2, cy + side / 2, side)


def cubist_textures(outline=dict(normal=HEAD_ON), outline_scale=1.15, views=dict(normal=(1, 0, 0), along=(0, -1, 2)),
                    jumble=(35, 35, 35), zoom=(0.5, 1.1), roll=25, background=True, seed=0, res=128):
    """What each face shows: a fragment, cut to the guitar's head-on outline.

    outline: where the guitar sits for its outline (its top facing normal, its neck toward along), at
      outline_scale, centered in the model. A face shows guitar only where a line from it straight
      toward the head-on viewer meets this guitar, with a dark contour at the edge. Head-on, the
      pieces line up into one outline.
    views: how a second guitar is turned for the fragments. The default faces its top along x and its
      neck up, so x sees its front, y its side, and z looks down along its neck.
    Each face's fragment is a close-up of one of FEATURES, chosen at random, seen along the face's own
      direction from directions(jumble, roll, seed), so roughly from its axis, framed so a face's edge
      spans between zoom[0] and zoom[1] of the guitar's units (the body is 2 across).
    background: outside the outline, show still_life() as it lies head-on, each face a slightly
      different tone so it reads as a plane. Otherwise a plain backdrop."""
    fs = faces()
    rng = np.random.default_rng(seed + 1)
    center = np.mean([c + (e1 + e2) / 2 for _, c, e1, e2 in fs], axis=0)
    out_frame = frame_from(outline.get("normal"), outline.get("along", HEAD_ON_UP))
    view_frame = frame_from(views.get("normal"), views.get("along", HEAD_ON_UP))
    g = (np.arange(res) + 0.5) / res
    names = list(FEATURES)
    if background:
        scene, (bx0, by0, bside) = still_life()
        _, r0, u0 = camera(0.0)
    out = []
    for (k, c, e1, e2), (d, rot) in zip(fs, directions(jumble, roll, seed)):
        P = c + g[None, :, None] * e1 + g[:, None, None] * e2      # row: along e2, column: along e1
        inside = covers(P, HEAD_ON, out_frame, center, outline_scale)
        # The fragment: the face's points as seen in its axis view, laid onto a view plane across d,
        # centered on a feature of the second guitar and rolled.
        _, rk, uk = camera(1.0, k)
        x, y = (P - c - (e1 + e2) / 2) @ rk, (P - c - (e1 + e2) / 2) @ uk
        up = uk - (uk @ d) * d; up /= np.linalg.norm(up)
        right = np.cross(up, d)
        right, up = np.cos(rot) * right + np.sin(rot) * up, np.cos(rot) * up - np.sin(rot) * right
        f = np.array(FEATURES[names[rng.integers(len(names))]]) @ np.array(view_frame)
        z = rng.uniform(*zoom)
        Q = f + z * (x[..., None] * right + y[..., None] * up)
        frag = cast(Q, d, view_frame, np.zeros(3), 1.0)
        missed = np.all(np.abs(frag - np.array(COLORS["background"])) < 1e-6, axis=-1)
        frag[missed] = COLORS["shadow"]
        if background:
            n = scene.shape[0]
            col = (((P @ r0) - bx0) / bside * n).astype(int).clip(0, n - 1)
            row = ((by0 - (P @ u0)) / bside * n).astype(int).clip(0, n - 1)
            back = scene[row, col] * rng.uniform(0.9, 1.08)
        else:
            back = np.array(COLORS["background"])
        tex = np.where(inside[..., None], frag, back)
        # The outline's contour.
        edge = np.zeros(inside.shape, bool)
        for a in (0, 1):
            e = np.diff(inside, axis=a)
            if a == 0:
                edge[:-1] |= e; edge[1:] |= e
            else:
                edge[:, :-1] |= e; edge[:, 1:] |= e
        tex[edge] = 0.12
        out.append(tex)
    return out

**The knobs.**

- `JUMBLE`: the most each axis's fragments tilt off their axis, in degrees (each tilts between half that and all of it, toward its own side). Higher drifts further from pure front, side and top views.
- `ZOOM`: how much of the guitar a face's edge spans, at random between the two (the body is 2 across). Lower is closer in.
- `ROLL`: the most each fragment turns in its face, in degrees.
- `BACKGROUND`: the still life outside the outline, or a plain backdrop if `False`.
- `SEED`: reshuffles the angles, rolls, the part each face shows, and the background planes' tones.

Saved as `cubism_guitar_3d.gif`.

In [ ]:
JUMBLE = (35, 35, 35)         # x, y, z
ZOOM = (0.5, 1.1)
ROLL = 25
BACKGROUND = True
SEED = 0

guitar_tex = cubist_textures(jumble=JUMBLE, zoom=ZOOM, roll=ROLL, background=BACKGROUND, seed=SEED)

fig, axes = plt.subplots(1, 4, figsize=(16, 4.5))
axes[0].imshow(render(textures=guitar_tex, size=400, light=LIT)); axes[0].set_title("head-on: the outline")
for k, name in enumerate(["x: front views", "y: side views", "z: views down the neck"]):
    axes[k + 1].imshow(render(textures=guitar_tex, t=1.0, axis=k, size=300)); axes[k + 1].set_title(name)
for ax in axes:
    ax.axis("off")
fig.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(7, 7))
ax.imshow(render(textures=guitar_tex, size=768, light=LIT)); ax.set_title("head-on"); ax.axis("off")
plt.show()

plot_tilts(None, ("x", "y", "z"), light=LIT, textures=guitar_tex)
plt.show()

from IPython.display import Image as ShowImage
save_gif(None, "cubism_guitar_3d.gif", light=LIT, textures=guitar_tex)
ShowImage(filename="cubism_guitar_3d.gif")

## Generate the three viewpoints (needs a GPU)

Plain DeepFloyd, one picture per viewpoint, both stages. The M models fit a T4 and take a minute or two for all three. Rerun the prompts cell and this one to try new prompts.

1. Use a **GPU runtime** (Runtime → Change runtime type → T4).
2. On Hugging Face, accept the licenses for [IF-I-M](https://huggingface.co/DeepFloyd/IF-I-M-v1.0) and [IF-II-M](https://huggingface.co/DeepFloyd/IF-II-M-v1.0), and have an [access token](https://huggingface.co/settings/tokens) ready.

To skip the GPU, set `pictures` to three of your own images instead: `[np.asarray(Image.open(f).convert("RGB"), float) / 255 for f in files]`, in x, y, z order.

In [ ]:
# protobuf 7.36 or later: current transformers ships a tokenizer file older protobuf can't read.
!pip install -q -U diffusers transformers accelerate sentencepiece "protobuf>=7.36"
from huggingface_hub import notebook_login
notebook_login()

**Prompts.** One per axis, sharing `STYLE` so the three read as one painting. Each picture is center-cropped to its window's box, so ask for the guitar to fill the picture.

In [ ]:
STYLE = "a cubist oil painting in the style of Picasso and Braque, ochre, brown and gray, of "
PROMPTS = (
    STYLE + "an acoustic guitar seen straight from the front, upright, filling the picture",   # x
    STYLE + "an acoustic guitar seen from the side in profile, upright, filling the picture",  # y
    STYLE + "an acoustic guitar seen from above, looking down along its neck",                 # z
)
TITLES = ("front", "side", "above")
SEED = 0

In [ ]:
import gc, torch
from transformers import T5EncoderModel
from diffusers import IFPipeline, IFSuperResolutionPipeline

# Load the big T5 text encoder straight onto the GPU, encode, and free it, before loading the image
# models. Loading everything at once fills Colab's RAM.
skip = dict(safety_checker=None, feature_extractor=None, watermarker=None)
text_encoder = T5EncoderModel.from_pretrained("DeepFloyd/IF-I-M-v1.0", subfolder="text_encoder", variant="fp16",
                                              torch_dtype=torch.float16, device_map="cuda")
encoder = IFPipeline.from_pretrained("DeepFloyd/IF-I-M-v1.0", text_encoder=text_encoder, unet=None, **skip)
with torch.no_grad():
    embeds = [encoder.encode_prompt(p) for p in PROMPTS]
del encoder, text_encoder
gc.collect(); torch.cuda.empty_cache()

if "stage_1" not in globals():
    stage_1 = IFPipeline.from_pretrained("DeepFloyd/IF-I-M-v1.0", text_encoder=None, variant="fp16",
                                         torch_dtype=torch.float16, **skip).to("cuda")
    stage_2 = IFSuperResolutionPipeline.from_pretrained("DeepFloyd/IF-II-M-v1.0", text_encoder=None, variant="fp16",
                                                        torch_dtype=torch.float16, **skip).to("cuda")

pictures = []
for pos, neg in embeds:
    gen = torch.Generator("cuda").manual_seed(SEED)
    small = stage_1(prompt_embeds=pos, negative_prompt_embeds=neg, generator=gen, output_type="pt").images
    big = stage_2(image=small, prompt_embeds=pos, negative_prompt_embeds=neg, generator=gen, output_type="pil").images[0]
    pictures.append(np.asarray(big.convert("RGB"), float) / 255)

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
for ax, pic, title in zip(axes, pictures, TITLES):
    ax.imshow(pic); ax.set_title(title); ax.axis("off")
plt.show()

## The demo

Head-on is the cubist jumble. Each row tilts toward one axis until its viewpoint resolves.

In [ ]:
plot_tilts(pictures, TITLES)
plt.show()

fig, ax = plt.subplots(figsize=(7, 7))
ax.imshow(render(pictures, 0.0, size=768)); ax.set_title("head-on"); ax.axis("off")
plt.show()

The same as an animation: head-on, then a tilt to each viewpoint and back. Saved as `cubism_guitar.gif`.

In [ ]:
from IPython.display import Image as ShowImage
save_gif(pictures, "cubism_guitar.gif")
ShowImage(filename="cubism_guitar.gif")

## What this leaves out

- **The real visibility map.** The demo paints ideal cube faces. The real model's faces come from scattered patches of the folded sheet, and its rim has partial faces, so we'll need the map from sheet to faces before printing.
- **Viewing at about 55°.** That's steep, and real cubes are rounded and imperfect, so the squares won't line up exactly.
- **Each picture is generated alone,** so the three don't know about each other, or about the cube edges that cut them. Next steps could be: a `STYLE` tuned until head-on reads as one painting, per-face prompts so each square is its own cubist plane (the whole-picture prompt controlling the low frequencies, as in arcimboldo), or a head-on view in the sampler if the jumble should read too.